# Conserver une mesure sans garder son graphe

Extraire des valeurs pour le journal de l’expérience.

Nous exécutons les cellules dans l’ordre, sur **CPU**. Aucun téléchargement de données ou de modèle n’est nécessaire.

Version de référence : **Python 3.11, PyTorch 2.10.0+cpu**. La cellule suivante indique la version réellement utilisée. Les sorties peuvent varier avec une autre version.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#detacher-geler-journaliser). Les deux exercices sont des textes à compléter : ils n’interrompent pas l’exécution intégrale. Leurs solutions restent repliées à la fin.

In [1]:
import platform
import torch

torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", platform.python_version())
print("PyTorch", torch.__version__)
print("Calcul", "CPU", "— threads", torch.get_num_threads())

Python 3.11.9
PyTorch 2.10.0+cpu
Calcul CPU — threads 1


## Nos repères

- **Journalisation (logging)** : La conservation des résultats pour observer l’expérience.
- **Scalaire (scalar)** : Une valeur unique.
- **Détachement (detach)** : Un accès aux valeurs sans le lien autograd précédent.
- **Gel (freezing)** : La désactivation des nouveaux gradients pour les paramètres choisis.

## Le calcul du cours

Avant de lancer cette cellule, annonçons les entrées et la sortie attendue. Après l’exécution, expliquons les nombres et les formes affichés. Le programme est identique à celui du chapitre.

In [2]:
import torch
w = torch.tensor(1., requires_grad=True)
loss = (w - 2.) ** 2
journal = loss.item()
scores_journal = torch.stack([loss, 2 * loss]).detach()
print("journal", journal, type(journal).__name__)
print("tableau", scores_journal.tolist())
print("suivi", scores_journal.requires_grad)
encodeur = torch.nn.Linear(3, 2)
for parametre in encodeur.parameters():
    parametre.requires_grad_(False)
encodeur.eval()
print("encodeur_gele", all(not p.requires_grad for p in encodeur.parameters()))

journal 1.0 float
tableau [1.0, 2.0]
suivi False
encodeur_gele True


## Nous vérifions notre compréhension

À dire en entretien : Je conserve les valeurs utiles sans garder leurs graphes. Je distingue ce détachement, la copie mémoire et le gel des paramètres.

## Exercice 1 — Conserver une mesure scalaire

Nous voulons un nombre Python dans le journal, sans conserver le graphe de ce calcul.

Remplaçons les points de suspension. Nous pouvons copier ce texte dans une nouvelle cellule de code.

```python
w = torch.tensor(1., requires_grad=True)
perte = (w - 2.).square()
mesure = perte....()
print(mesure, type(mesure).__name__)
```

## Exercice 2 — Apprendre seulement la tête

Gelons l’encodeur avant le calcul. Nous vérifierons que la tête reçoit encore des gradients.

Remplaçons les points de suspension. Nous pouvons copier ce texte dans une nouvelle cellule de code.

```python
encodeur = torch.nn.Linear(2, 2)
tete = torch.nn.Linear(2, 1)
for p in encodeur.parameters():
    p.requires_grad_(...)
perte = tete(encodeur(torch.tensor([[1., 2.]]))).square().mean()
perte.backward()
print([p.grad is None for p in encodeur.parameters()])
```

# Solutions

Nous cherchons d’abord, puis ouvrons la solution utile. Chaque programme ci-dessous est autonome. Le vérificateur du dépôt exécute également ces solutions, séparément des exercices à trous.

<details>
<summary>Afficher la solution de l’exercice 1 — Conserver une mesure scalaire</summary>

<pre><code>import torch
w = torch.tensor(1., requires_grad=True)
perte = (w - 2.).square()
mesure = perte.item()
assert isinstance(mesure, float)
print(mesure, type(mesure).__name__)</code></pre>
</details>

<details>
<summary>Afficher la solution de l’exercice 2 — Apprendre seulement la tête</summary>

<pre><code>import torch
torch.manual_seed(17)
encodeur = torch.nn.Linear(2, 2)
tete = torch.nn.Linear(2, 1)
for p in encodeur.parameters():
    p.requires_grad_(False)
perte = tete(encodeur(torch.tensor([[1., 2.]]))).square().mean()
perte.backward()
assert all(p.grad is None for p in encodeur.parameters())
assert all(p.grad is not None for p in tete.parameters())
print([p.grad is None for p in encodeur.parameters()])</code></pre>
</details>